# Capítulo 3 · Algoritmo de Deutsch-Jozsa

## Objetivos

1. Entender el modelo de computación basado en **oráculos** y el problema que resuelve Deutsch-Jozsa.
2. Construir los oráculos constantes y balanceados para $n$ bits.
3. Implementar el algoritmo completo en Qiskit y verificar la separación cuántico-clásica.

---

## 3.1 El problema de Deutsch-Jozsa

Dado un oráculo que computa una función $f: \{0,1\}^n \to \{0,1\}$, se garantiza que $f$ es:

- **Constante**: $f(x) = 0$ o $f(x) = 1$ para todo $x$, o
- **Balanceada**: $f(x) = 0$ exactamente para la mitad de las entradas y $f(x) = 1$ para la otra mitad.

Un algoritmo clásico necesita $2^{n-1} + 1$ evaluaciones en el peor caso. El algoritmo cuántico de Deutsch-Jozsa determina la respuesta con **una sola llamada al oráculo**.

El algoritmo aplica la secuencia:

$$|0\rangle^{\otimes n}|1\rangle \xrightarrow{H^{\otimes n+1}} |+\rangle^{\otimes n}|-\rangle \xrightarrow{U_f} \cdots \xrightarrow{H^{\otimes n}} \text{medir}$$

Si el resultado es $|00\ldots 0\rangle$, $f$ es constante; en caso contrario, es balanceada.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from src.visualization import QuantumVisualization

print('Qiskit y módulos locales cargados.')

## 3.2 Oráculos para Deutsch-Jozsa

In [ ]:
def oracle_constant_0(n: int) -> QuantumCircuit:
    """Oráculo para f(x) = 0 (constante). No hace nada."""
    return QuantumCircuit(n + 1, name='Uf(const-0)')


def oracle_constant_1(n: int) -> QuantumCircuit:
    """Oráculo para f(x) = 1 (constante). Aplica X al qubit ancilla."""
    qc = QuantumCircuit(n + 1, name='Uf(const-1)')
    qc.x(n)
    return qc


def oracle_balanced(n: int, balanced_string: str | None = None) -> QuantumCircuit:
    """Oráculo para una función balanceada.

    La función f(x) = CNOT basado en un bitstring: aplica CNOTs desde
    cada qubit de entrada cuyo bit sea '1' en balanced_string hacia la ancilla.
    Si balanced_string es None, se usa '1'*n (todos activos).

    Parámetros
    ----------
    n : int
        Número de qubits de entrada.
    balanced_string : str, optional
        Cadena binaria de longitud n. '1' en posición i activa la CNOT.
    """
    if balanced_string is None:
        balanced_string = '1' * n
    assert len(balanced_string) == n, 'balanced_string debe tener longitud n'

    qc = QuantumCircuit(n + 1, name='Uf(balanced)')
    for i, bit in enumerate(balanced_string):
        if bit == '1':
            qc.cx(i, n)
    return qc


# Visualizar los oráculos para n=3
n = 3
print('Oráculo constante 0:')
print(oracle_constant_0(n).draw('text'))
print('\nOráculo constante 1:')
print(oracle_constant_1(n).draw('text'))
print('\nOráculo balanceado (string=101):')
print(oracle_balanced(n, '101').draw('text'))

## 3.3 Algoritmo completo de Deutsch-Jozsa

In [ ]:
def deutsch_jozsa(oracle: QuantumCircuit, n: int) -> QuantumCircuit:
    """Construye el circuito completo de Deutsch-Jozsa.

    Parámetros
    ----------
    oracle : QuantumCircuit
        Circuito del oráculo U_f (n+1 qubits).
    n : int
        Número de qubits de entrada.

    Retorna
    -------
    QuantumCircuit
        Circuito completo listo para ejecutar.
    """
    qc = QuantumCircuit(n + 1, n)

    # Inicialización: ancilla en |1〉
    qc.x(n)
    qc.barrier(label='Init')

    # Hadamard sobre todos los qubits
    qc.h(range(n + 1))
    qc.barrier(label='H⊗(n+1)')

    # Oráculo
    qc.compose(oracle, inplace=True)
    qc.barrier(label='Oráculo')

    # Segunda capa de Hadamard sobre los qubits de entrada
    qc.h(range(n))
    qc.barrier(label='H⊗n')

    # Medida de los qubits de entrada
    qc.measure(range(n), range(n))

    return qc


# Instanciar con los oráculos
n = 4
backend = AerSimulator()

for oracle_fn, oracle_name in [
    (oracle_constant_0(n), 'Constante 0'),
    (oracle_constant_1(n), 'Constante 1'),
    (oracle_balanced(n, '1011'), 'Balanceada (1011)'),
    (oracle_balanced(n, '0110'), 'Balanceada (0110)'),
]:
    qc = deutsch_jozsa(oracle_fn, n)
    job = backend.run(qc, shots=1024)
    counts = job.result().get_counts()
    all_zeros = '0' * n
    decision = 'CONSTANTE' if all_zeros in counts and counts[all_zeros] == 1024 else 'BALANCEADA'
    print(f'Oráculo: {oracle_name:25s} → Decisión: {decision}')
    print(f'  Conteos: {counts}')

In [ ]:
# Visualización del circuito para n=3
n = 3
qc_example = deutsch_jozsa(oracle_balanced(n, '110'), n)
print(f'Circuito Deutsch-Jozsa, n={n}, oráculo balanceado (110):')
print(qc_example.draw('text'))

## 3.4 Escala y ventaja cuántica

El siguiente análisis empírico muestra que el algoritmo cuántico siempre necesita exactamente **una** evaluación del oráculo, independientemente de $n$.

In [ ]:
import random

classical_queries = []
quantum_queries   = []
n_values = range(1, 12)

for n in n_values:
    # Clásico (peor caso)
    classical_queries.append(2 ** (n - 1) + 1)
    # Cuántico: siempre 1
    quantum_queries.append(1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(n_values), classical_queries, 'o-', color='#f78166',
        label='Clásico (peor caso)', linewidth=2)
ax.plot(list(n_values), quantum_queries, 's-', color='#58a6ff',
        label='Cuántico (Deutsch-Jozsa)', linewidth=2)
ax.set_xlabel('Número de bits de entrada (n)')
ax.set_ylabel('Evaluaciones del oráculo')
ax.set_title('Ventaja cuántica: Deutsch-Jozsa vs. algoritmo clásico')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 3.5 Ejercicios propuestos

1. Implementa el caso $n=1$ (Deutsch original) manualmente con matrices NumPy, sin Qiskit. Verifica el resultado.

2. ¿Qué ocurre si el oráculo $U_f$ introduce errores (bit-flip con probabilidad $p = 0.01$)? Modifica el circuito para incluir un canal de ruido y analiza la tasa de error del algoritmo.

3. Construye un oráculo balanceado para $n=5$ en el que exactamente la mitad de las cadenas de entrada mapean a 1. Verifica con el algoritmo que se detecta correctamente.

4. Demuestra formalmente (con cálculo de estados) que midiendo $|00\ldots 0\rangle$ con probabilidad 1 implica que $f$ es constante.